# ЕГЖ РСИИ — нұсқа 3: Қант диабетінің қаупі (Pima Indians Diabetes)
EDA, модельдерді салыстыру, бағалау, интерпретация, сақтау.

## 0. Дайындық
Google Colab-та `project.zip` жүктеңіз (кодтың қалған бөлігі `src/` ішінде). Деректер: `diabetes.csv` (Kaggle) — болмаса сілтемеден жүктеледі.

In [ ]:
import os, sys, zipfile, warnings
warnings.filterwarnings("ignore")
try:
    from google.colab import files
    if not os.path.exists("src"):
        up = files.upload()                      # project.zip
        zipfile.ZipFile("project.zip").extractall(".")
except ImportError:
    pass
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                               # локально: жоба түбіріне өту
sys.path.insert(0, os.getcwd())
os.makedirs("models", exist_ok=True)

## 1. Деректерді жүктеу

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
URL = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
df = pd.read_csv("data/diabetes.csv" if os.path.exists("data/diabetes.csv") else URL)
print(df.shape); df.head()

## 2. EDA

In [ ]:
print(df.dtypes.to_string()); print("\nБос мәндер (NaN):", df.isna().sum().sum())
df.describe().T

In [ ]:
cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
print("Жасырын бос мәндер (0 түрінде):"); print((df[cols] == 0).sum().to_string())

**Қорытынды:** NaN жоқ, бірақ `Glucose, BloodPressure, SkinThickness, Insulin, BMI` бағандарында 0 мәндері бар — физиологиялық мүмкін емес, яғни жасырын бос мәндер («қиын жер»). Insulin мен SkinThickness-те олар өте көп.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
(df[cols] == 0).sum().plot.bar(ax=ax[0], color="tomato"); ax[0].set_title("График 1. «0» түріндегі жасырын бос мәндер")
df["Outcome"].value_counts().plot.bar(ax=ax[1]); ax[1].set_title("График 2. Кластардың таралуы"); plt.tight_layout(); plt.show()
print(df["Outcome"].value_counts(normalize=True).round(3).to_dict())

**Қорытынды:** класс теңгерімсіздігі орташа (~65% / 35%), сондықтан Accuracy емес, Recall мен ROC-AUC қолданамыз, `class_weight='balanced'` қосамыз.

In [ ]:
d = df.copy(); d[cols] = d[cols].replace(0, np.nan)
d.drop(columns="Outcome").hist(figsize=(11, 6), bins=20); plt.suptitle("График 3. Белгілердің таралуы (0 → NaN)"); plt.tight_layout(); plt.show()

**Қорытынды:** Insulin, DiabetesPedigreeFunction, Age оңға қарай қиғаш (ұзын құйрық) — шектен тыс мәндер бар; медиана арқылы толтыру орташадан дұрысырақ.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for a, c in zip(ax, ["Glucose", "BMI", "Age"]):
    sns.boxplot(data=d, x="Outcome", y=c, ax=a)
ax[1].set_title("График 4. Белгілер мен нәтиже (Outcome)"); plt.tight_layout(); plt.show()

**Қорытынды:** диабеті барларда Glucose айқын жоғары, BMI мен Age де жоғары — ең күшті ажырататын белгі Glucose.

In [ ]:
plt.figure(figsize=(7, 5)); sns.heatmap(d.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("График 5. Корреляция матрицасы"); plt.show()

**Қорытынды:** Outcome-пен ең жоғары корреляция — Glucose (0.49), одан кейін BMI (0.31) және Insulin (0.30). Insulin–Glucose және SkinThickness–BMI арасында байланыс бар (толтыру үшін пайдалы).

## 3. Алдын ала өңдеу және модельдерді салыстыру
Барлық өңдеу (0→NaN, медиана, масштабтау) **Pipeline** ішінде, сондықтан кросс-валидацияда деректердің ағып кетуі жоқ.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from src.preprocessing import zeros_to_nan

FEATURES = [c for c in df.columns if c != "Outcome"]
X, y = df[FEATURES], df["Outcome"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

def make(model):
    return Pipeline([("zeros", FunctionTransformer(zeros_to_nan)), ("imputer", SimpleImputer(strategy="median")),
                     ("scaler", StandardScaler()), ("model", model)])

models = {
    "LogisticRegression": (make(LogisticRegression(max_iter=1000, class_weight="balanced")), {"model__C": [0.01, 0.1, 1, 10]}),
    "RandomForest": (make(RandomForestClassifier(random_state=42, class_weight="balanced")), {"model__n_estimators": [100, 300], "model__max_depth": [3, 5, None]}),
    "GradientBoosting": (make(GradientBoostingClassifier(random_state=42)), {"model__n_estimators": [100, 200], "model__learning_rate": [0.03, 0.1], "model__max_depth": [2, 3]}),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows, best = [], {}
for name, (pipe, grid) in models.items():
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc").fit(X_tr, y_tr)
    best[name] = gs.best_estimator_
    rec = cross_val_predict(gs.best_estimator_, X_tr, y_tr, cv=cv).astype(bool)
    rows.append({"Модель": name, "Үздік параметрлер": gs.best_params_, "CV ROC-AUC": round(gs.best_score_, 3),
                 "CV Recall": round((rec & (y_tr == 1)).sum() / (y_tr == 1).sum(), 3)})
pd.DataFrame(rows)

**Қорытынды:** үш модельдің ROC-AUC көрсеткіші бір-біріне жақын. Логистикалық регрессия — ең қарапайым, түсіндіруге ыңғайлы (коэффициенттер) және сараптамалық жүйемен үйлеседі, сондықтан соңғы модель ретінде таңдалады, тек егер ол басқалардан айтарлықтай нашар болмаса.

## 4. Шекті мәнді (threshold) таңдау
Диабетті «байқамай қалу» (False Negative) қымбат, сондықтан Recall басым. Шекті мәнді **тек train-нің кросс-валидациялық болжамдарынан** таңдаймыз (test-ке қарамаймыз): Recall ≥ 0.85 болатын ең жоғары шек.

In [ ]:
final = best["LogisticRegression"]
proba_cv = cross_val_predict(final, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
from sklearn.metrics import recall_score, precision_score
ths = np.arange(0.1, 0.9, 0.01)
rec = np.array([recall_score(y_tr, proba_cv >= t) for t in ths])
prec = np.array([precision_score(y_tr, proba_cv >= t) for t in ths])
THR = float(ths[rec >= 0.85].max())
plt.plot(ths, rec, label="Recall"); plt.plot(ths, prec, label="Precision"); plt.axvline(THR, ls="--", c="k")
plt.xlabel("threshold"); plt.legend(); plt.title(f"Таңдалған шек = {THR:.2f}"); plt.show()
THR

## 5. Test жиынында бағалау

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score
final.fit(X_tr, y_tr)
proba = final.predict_proba(X_te)[:, 1]; pred = (proba >= THR).astype(int)
print(classification_report(y_te, pred, target_names=["Дені сау", "Диабет қаупі"]))
print("ROC-AUC:", round(roc_auc_score(y_te, proba), 3))
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["Сау", "Диабет"]); plt.show()

**Қорытынды:** Recall жоғары (диабетті сирек жіберіп аламыз), ал Precision төмендеу — «бекер дабыл» көбірек. Скринингте бұл қолайлы айырбас: артық тексеру аурудың өтіп кетуінен арзан.

## 6. Интерпретация

In [ ]:
from sklearn.inspection import permutation_importance
pi = permutation_importance(final, X_te, y_te, scoring="roc_auc", n_repeats=20, random_state=42)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(title="Permutation importance (ROC-AUC)"); plt.show()

**Қорытынды:** ең маңызды белгі — Glucose (айырмашылығы өте үлкен), одан кейін BMI мен Pregnancies; бұл медициналық білімге сәйкес келеді.

In [ ]:
err = X_te.copy(); err["Outcome"] = y_te; err["proba"] = proba.round(3); err["pred"] = pred
err["error"] = (err["proba"] - err["Outcome"]).abs()
err.sort_values("error", ascending=False).head(5)

**Қорытынды:** ең ауыр қателердің көбі — «жалған дабыл» (False Positive): Glucose өте жоғары (165–194), бірақ белгісі «сау» — модель Glucose-ке көбірек сүйенеді. Бір False Negative-те (Glucose = 78) диабет бар, бірақ мәндер қалыпты — модель мұндай жағдайды ұстай алмайды. Қосымша себеп: Insulin/SkinThickness жасырын бос болып, медианамен толтырылған.

## 7. Модельді сақтау
Файл `src/decision.py` күтетін форматта: `pipeline`, `threshold`, `features`. Соңында тесттерді жүргіземіз.

In [ ]:
import joblib
final.fit(X, y)                                  # соңғы модель — барлық деректерде
joblib.dump({"pipeline": final, "threshold": THR, "features": FEATURES}, "models/model.joblib")
print("Сақталды, шек =", round(THR, 3))

In [ ]:
!pip -q install pytest httpx fastapi
!python -m pytest -q

## 8. API-ды Colab-та тексеру (Swagger орнына)

In [ ]:
from fastapi.testclient import TestClient
from api.main import app
c = TestClient(app)
print(c.get("/health").json())
ok = {"Pregnancies":1,"Glucose":95,"BloodPressure":70,"SkinThickness":20,"Insulin":80,"BMI":23,"DiabetesPedigreeFunction":0.3,"Age":28}
print(c.post("/predict", json=ok).json())
print(c.post("/explain", json={**ok, "Glucose": 150, "BMI": 32}).json())
print(c.post("/predict", json={**ok, "Age": 250}).status_code)